In [ ]:
!pip install -q transformers datasets peft accelerate trl bitsandbytes
!pip install -q transformers datasets peft accelerate torch


import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from tqdm import tqdm
from transformers import get_scheduler
from peft import LoraConfig
from trl import SFTTrainer
from huggingface_hub import login
from torch.utils.data import DataLoader
import time


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.9 MB/s eta 0:00:00


In [ ]:
!pip install datasets


In [ ]:
from datasets import load_dataset

Dataset = load_dataset("spider")
print(Dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [ ]:
from huggingface_hub import login
login(new_session=False)

In [ ]:
# ----------------------------
# Step 1: Hugging Face Login
# ----------------------------
HF_TOKEN = "----------------------"   # replace with your token
login(HF_TOKEN)

# ----------------------------
# Step 2: Load Tokenizer & Model
# ----------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_8bit=True,  # memory efficient, works on small GPU
    device_map="auto",
    token=HF_TOKEN
)

# ----------------------------
# Step 3: LoRA Config
# ----------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],  # key attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# ----------------------------
# Step 4: Load Spider Dataset
# ----------------------------
dataset = load_dataset("spider")

# ----------------------------
# Step 5: Format Spider -> Prompt/Response
# ----------------------------
def format_prompt(example):
    # prompt: natural language → SQL
    prompt = f"Translate the following question into SQL:\nQuestion: {example['question']}\nSQL:"
    return {
        "input_text": prompt,
        "target_text": example["query"]
    }

train_dataset = dataset["train"].map(format_prompt)
val_dataset = dataset["validation"].map(format_prompt)

# ----------------------------
# Step 6: Torch Dataset Class
# ----------------------------
from torch.utils.data import Dataset # Import Dataset from torch.utils.data

class SpiderText2SQLDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=512):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        enc = self.tokenizer(
            item["input_text"],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        dec = self.tokenizer(
            item["target_text"],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].squeeze()
        attention_mask = enc["attention_mask"].squeeze()
        labels = dec["input_ids"].squeeze()

        # Mask padding tokens in labels
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

train_torch = SpiderText2SQLDataset(train_dataset, tokenizer)
val_torch = SpiderText2SQLDataset(val_dataset, tokenizer)

train_loader = DataLoader(train_torch, batch_size=2, shuffle=True)
val_loader = DataLoader(val_torch, batch_size=2)

# ----------------------------
# Step 7: Optimizer & Scheduler
# ----------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 1  # 1 epoch for demo
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer,
    num_warmup_steps=100,
    num_training_steps=num_training_steps
)

# ----------------------------
# Step 8: Training Loop
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

num_epochs = 1
gradient_accumulation_steps = 4
step = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    total_loss = 0


    # Track epoch start time
    epoch_start = time.time()
    total_steps = len(train_loader)

    for batch_idx, batch in enumerate(train_loader, start=1):
        step += 1

        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)


        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / gradient_accumulation_steps  # Normalize loss
        total_loss += loss.item() * gradient_accumulation_steps

        # Backward pass
        loss.backward()

        # Gradient accumulation
        if step % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        # Compute ETA for entire epoch (based on elapsed time per step so far)
        elapsed = time.time() - epoch_start
        avg_time_per_step = elapsed / batch_idx
        steps_left = total_steps - batch_idx
        eta_epoch = avg_time_per_step * steps_left

        # Print step loss
        print(f"Epoch {epoch+1} | Step {batch_idx}/{total_steps} |" f"Loss: {loss.item() * gradient_accumulation_steps:.4f} | \n"
              f"ETA (Epoch): {eta_epoch/60:.2f} min")

    # Epoch average loss
    avg_loss = total_loss / len(train_loader)
    print(f"✅ Epoch {epoch + 1} completed. Average Loss: {avg_loss:.4f}")

# ----------------------------
# Step 9: Save Fine-tuned LoRA
# ----------------------------
save_dir = "./qwen_spider_lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"✅ Model saved at {save_dir}")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



Epoch 1/1


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch 1 | Step 1/3500 |Loss: 14.8240 | 
ETA (Epoch): 260.49 min
Epoch 1 | Step 2/3500 |Loss: 14.4643 | 
ETA (Epoch): 224.37 min
Epoch 1 | Step 3/3500 |Loss: 14.4971 | 
ETA (Epoch): 211.64 min
Epoch 1 | Step 4/3500 |Loss: 14.7535 | 
ETA (Epoch): 207.34 min
Epoch 1 | Step 5/3500 |Loss: 14.6832 | 
ETA (Epoch): 203.24 min
Epoch 1 | Step 6/3500 |Loss: 14.4858 | 
ETA (Epoch): 201.31 min
Epoch 1 | Step 7/3500 |Loss: 14.9170 | 
ETA (Epoch): 199.35 min
Epoch 1 | Step 8/3500 |Loss: 14.4844 | 
ETA (Epoch): 198.00 min
Epoch 1 | Step 9/3500 |Loss: 14.2441 | 
ETA (Epoch): 197.10 min
Epoch 1 | Step 10/3500 |Loss: 14.9436 | 
ETA (Epoch): 196.41 min
Epoch 1 | Step 11/3500 |Loss: 14.5009 | 
ETA (Epoch): 195.84 min
Epoch 1 | Step 12/3500 |Loss: 15.1972 | 
ETA (Epoch): 195.50 min
Epoch 1 | Step 13/3500 |Loss: 15.2597 | 
ETA (Epoch): 195.53 min
Epoch 1 | Step 14/3500 |Loss: 14.7414 | 
ETA (Epoch): 195.31 min
Epoch 1 | Step 15/3500 |Loss: 14.1742 | 
ETA (Epoch): 195.18 min
Epoch 1 | Step 16/3500 |Loss: 14.9

In [ ]:
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --- Load base model in 4-bit for efficiency (works on 8GB GPU) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading base Qwen model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# --- Load LoRA fine-tuned adapters ---
print("Loading fine-tuned LoRA adapters...")
ft_model = PeftModel.from_pretrained(base_model, "./qwen_spider_lora")  # your output dir

ft_model.eval()

# --- SQL Generation Function ---
def generate_sql(question: str, schema: str, max_new_tokens: int = 256):
    """
    Given a natural language question, generate SQL + natural language response.
    """
    prompt = f"""
You are an expert SQL assistant.
Convert the following user request into a valid PostgreSQL query,
and then explain the result in plain English.
You must ONLY use the provided schema to write queries.
If a table or column is not listed, you must NOT invent it.
Always return only the SQL query, nothing else.
You must strictly follow the schema provided.
Never invent tables or columns. Only return SQL wrapped in ```sql ... ``` blocks.


User question: {question}

SQL Query:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=False
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded[len(prompt):].strip()

    return response


# --- Example Inference ---
#if __name__ == "__main__":


#user_q = "What are the names of all students enrolled in the Computer Science department?"
#print("\n[User]:", user_q)
#answer = generate_sql(user_q)
#print("\n[Model]:", answer)

In [ ]:
user_q= "How many students are there in computer science department?"
print("\n[User]:", user_q)
answer = generate_sql(user_q)
print("\n[Model]:", answer)